# Real-Time Weather Data Producer with Apache Kafka


## Overview

This notebook implements the **data producer** component of the real-time energy prediction pipeline. It simulates a live weather sensor feed by reading historical records from a CSV file and streaming them to an Apache Kafka topic at configurable intervals.

The downstream notebook (`03_spark_streaming_prediction.ipynb`) subscribes to this topic, applies the trained ML pipeline from notebook 01, and writes predictions to Parquet.

### Pipeline Position

```
data/weather.csv --> [02 Kafka Producer] --> Kafka Topic: weather-stream --> [03 Spark Streaming]
                                                                                      ^
                                                                          models/energy_pipeline_model
                                                                          (trained in Notebook 01)
```

### How It Works

1. Loads all records from `data/weather.csv` sorted by timestamp
2. Every **5 seconds**, publishes a batch of **5 days of data** (120 records) to the `weather-stream` Kafka topic
3. Each day within the batch is assigned a distinct Unix timestamp to preserve event ordering for the stream consumer
4. The file pointer advances with each batch, consuming the full dataset chronologically

### Prerequisites

- Apache Kafka broker running at `kafka:9092` (start via Docker - see the project README)
- `kafka-python` package (pinned in requirements.txt: `kafka-python==2.0.2`)
- Notebook 03 running and subscribed to the `weather-stream` topic


## 1. Imports and Configuration

Import the required libraries and set the Kafka broker address, port, and target topic name.


In [ ]:
from time import sleep
from json import dumps
from kafka import KafkaProducer
import csv
import datetime as dt
import time

# Kafka broker address and target topic
KAFKA_HOST = "localhost"
KAFKA_PORT = 9092
TOPIC = "weather-stream"

## 2. Helper Functions

Three utility functions handle the core producer operations:

| Function | Description |
|----------|-------------|
| `read_csv(file_path)` | Reads and sorts all weather records by timestamp |
| `publish_message(producer, topic, data)` | Serialises a payload as JSON and sends it to the specified Kafka topic |
| `connect_kafka_producer()` | Establishes a connection to the Kafka broker and returns a `KafkaProducer` instance |


In [ ]:
def read_csv(file_path):
    """Read and sort weather records from CSV by timestamp."""
    rows = []
    with open(file_path, "rt") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    rows.sort(key=lambda r: r["timestamp"])
    return rows


def publish_message(producer, topic_name, data):
    """Serialise data as JSON and publish it to the specified Kafka topic."""
    try:
        producer.send(topic_name, value=data)
        print("Message published successfully.")
    except Exception as ex:
        print(f"Failed to publish message: {ex}")


def connect_kafka_producer():
    """Create and return a KafkaProducer connected to the configured broker."""
    try:
        producer = KafkaProducer(
            bootstrap_servers=[f"{KAFKA_HOST}:{KAFKA_PORT}"],
            value_serializer=lambda x: dumps(x).encode("utf-8"),
            api_version=(0, 10),
        )
        print("Connected to Kafka broker successfully.")
        return producer
    except Exception as ex:
        print(f"Failed to connect to Kafka: {ex}")
        return None

## 3. Producer Loop

Iterates through the weather dataset in 5-day batches. Each batch is timestamped with the current wall-clock time (one Unix second per day) before being published to the Kafka topic. The loop waits 5 seconds between batches to simulate real-time data ingestion.

Interrupt the kernel (stop button or `Ctrl+C`) to shut down the producer gracefully.


In [ ]:
import os

RECORDS_PER_DAY = 24
DAYS_PER_BATCH = 5
BATCH_SIZE = RECORDS_PER_DAY * DAYS_PER_BATCH

# Optional cap for a bounded/headless demo run. Leave PRODUCER_MAX_BATCHES unset
# (or 0) to keep the original unbounded behaviour (drains the full dataset).
MAX_BATCHES = int(os.environ.get("PRODUCER_MAX_BATCHES", "0")) or None

print("Starting Weather Data Producer...")
producer = connect_kafka_producer()
rows = read_csv("data/weather.csv")

days_sent = 0
batch_count = 1

try:
    while days_sent < len(rows):
        if MAX_BATCHES and batch_count > MAX_BATCHES:
            print(f"Reached PRODUCER_MAX_BATCHES={MAX_BATCHES}; stopping.")
            break

        batch = rows[days_sent : days_sent + BATCH_SIZE]
        if not batch:
            print("All records have been sent.")
            break

        base_ts = int(time.time())

        # Assign a distinct Unix timestamp to each day in the batch
        for day_index in range(DAYS_PER_BATCH):
            start = day_index * RECORDS_PER_DAY
            end = start + RECORDS_PER_DAY
            for j in range(start, min(end, len(batch))):
                batch[j]["weather_ts"] = base_ts + day_index

        payload = {
            "ts": dt.datetime.now().isoformat(timespec="seconds"),
            "rows": batch,
        }

        publish_message(producer, TOPIC, payload)
        print(
            f"Batch {batch_count}: sent {len(batch)} records "
            f"(weather_ts {base_ts} to {base_ts + DAYS_PER_BATCH - 1}). "
            f"Next batch in 5 seconds."
        )

        days_sent += BATCH_SIZE
        batch_count += 1
        sleep(5)

except KeyboardInterrupt:
    print("Producer stopped by user.")

finally:
    if producer:
        producer.close()
    print("Kafka producer connection closed.")